In [1]:
import matplotlib.pyplot as plt
import xarray as xr
from sklearn.metrics import r2_score
from pathlib import Path
import pandas as pd
import os

In [2]:
# ------------ Paths ------------
basins_path = Path("./531_basin.txt")
original_caravan_path = Path("/inputs/data_updated_2/time_series")
camels_timeseries = Path("/inputs/basin_dataset_public_v1p2/basin_mean_forcing/daymet") #US

In [3]:
with open(basins_path, "r") as f:
    basins = f.read().splitlines()
basins

['camels_01022500',
 'camels_01031500',
 'camels_01047000',
 'camels_01052500',
 'camels_01054200',
 'camels_01055000',
 'camels_01057000',
 'camels_01073000',
 'camels_01078000',
 'camels_01123000',
 'camels_01134500',
 'camels_01137500',
 'camels_01139000',
 'camels_01139800',
 'camels_01142500',
 'camels_01144000',
 'camels_01162500',
 'camels_01169000',
 'camels_01170100',
 'camels_01181000',
 'camels_01187300',
 'camels_01195100',
 'camels_04296000',
 'camels_01333000',
 'camels_01350000',
 'camels_01350080',
 'camels_01350140',
 'camels_01365000',
 'camels_01411300',
 'camels_01413500',
 'camels_01414500',
 'camels_01415000',
 'camels_01423000',
 'camels_01434025',
 'camels_01435000',
 'camels_01439500',
 'camels_01440000',
 'camels_01440400',
 'camels_01451800',
 'camels_01466500',
 'camels_01484100',
 'camels_01487000',
 'camels_01491000',
 'camels_01510000',
 'camels_01516500',
 'camels_01518862',
 'camels_01532000',
 'camels_01539000',
 'camels_01542810',
 'camels_01543000',


In [4]:
for basin_name in basins:
    basin_id = basin_name.replace("camels_", "")

    try:
        # --- Load nc file ---
        nc_path = original_caravan_path/f"{basin_name}.nc"
        ds = xr.open_dataset(nc_path)

        # --- Load CAMELS txt ---
        file_path = next(camels_timeseries.rglob(f"*{basin_id}*.txt"))
        camels = pd.read_csv(file_path, sep=r"\s+", skiprows=3)

        # Build datetime
        camels["date"] = pd.to_datetime(
            camels[["Year", "Mnth", "Day"]]
            .rename(columns={"Year": "year", "Mnth": "month", "Day": "day"})
        )

        # --- Create DataArray ---
        camels_da = xr.DataArray(
            data=camels["prcp(mm/day)"].values,
            dims=["date"],
            coords={"date": camels["date"].values},
        )

        # --- Align to nc dates ONLY ---
        camels_da = camels_da.reindex(date=ds["date"])

        # --- Add to dataset ---
        ds["camels_precipitation"] = camels_da

        # --- Safe save ---
        tmp_path = nc_path.with_suffix(".tmp")
        ds.to_netcdf(tmp_path)
        ds.close()
        os.replace(tmp_path, nc_path)

        print(f"Added camels_precipitation to {basin_name}")

    except Exception as e:
        print(f"Failed for {basin_name}: {e}") 

Added camels_precipitation to camels_01022500
Added camels_precipitation to camels_01031500
Added camels_precipitation to camels_01047000
Added camels_precipitation to camels_01052500
Added camels_precipitation to camels_01054200
Added camels_precipitation to camels_01055000
Added camels_precipitation to camels_01057000
Added camels_precipitation to camels_01073000
Added camels_precipitation to camels_01078000
Added camels_precipitation to camels_01123000
Added camels_precipitation to camels_01134500
Added camels_precipitation to camels_01137500
Added camels_precipitation to camels_01139000
Added camels_precipitation to camels_01139800
Added camels_precipitation to camels_01142500
Added camels_precipitation to camels_01144000
Added camels_precipitation to camels_01162500
Added camels_precipitation to camels_01169000
Added camels_precipitation to camels_01170100
Added camels_precipitation to camels_01181000
Added camels_precipitation to camels_01187300
Added camels_precipitation to came